## Active Users

In [1]:
import os
from pathlib import Path

from dotenv import find_dotenv, load_dotenv
from google.cloud import bigquery

load_dotenv(find_dotenv(usecwd=True))
project_id = os.environ.get("GOOGLE_CLOUD_PROJECT")
if not project_id:
    raise RuntimeError("Set GOOGLE_CLOUD_PROJECT in a .env file (see ../.env.example).")

# Queries are billed to (or, in the sandbox, counted against) your own project, not the dataset's.
client = bigquery.Client(project=project_id)

SQL_DIR = Path("sql")
RAW_DIR = Path("csv")


def run_query(name):
    """Run sql/<name>.sql and return the result."""
    sql = (SQL_DIR / f"{name}.sql").read_text(encoding="utf-8")
    # A dry run is free and shows the cost before the real run.
    estimate = client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True))
    print(f"{name}: {estimate.total_bytes_processed / 1024**3:.2f} GB to read")
    return client.query(sql).to_dataframe()


def save(df, name):
    # LF line endings on every OS, so a re-run matches the committed file byte for byte.
    df.to_csv(RAW_DIR / f"{name}.csv", index=False, encoding="utf-8", lineterminator="\n")

In [2]:
parts = run_query("active_users_20201101")
parts

active_users_20201101: 0.01 GB to read


,part,users
0,1 first_visit,1946
1,2 engaged session,236
2,3 engagement_time_msec,59
3,4 not active,124


In [3]:
total = run_query("total_users_20201101")
total

total_users_20201101: 0.00 GB to read


,total_users
0,2365


In [4]:
total_users = total.total_users.item()

print(parts)
print("Sum of the four parts:", parts.users.sum())
print("Total users, counted directly:", total_users)

assert parts.users.tolist() == [1946, 236, 59, 124], "The four parts differ from the page"
assert parts.users.sum() == total_users, "The sum of the four parts is not the day's total users"

                     part  users
0           1 first_visit   1946
1       2 engaged session    236
2  3 engagement_time_msec     59
3            4 not active    124
Sum of the four parts: 2365
Total users, counted directly: 2365


In [5]:
save(parts, "active_users_20201101")